# doc_index 점검 노트북

인덱스 결과를 눈으로 확인하는 보조용입니다. 판정·변환 로직은 `payment_card/file_pipeline/doc_index/*.py`에만 있고, 여기서는 불러와 보여 주기만 합니다.

- 커밋 전에 **모든 셀 출력을 지웁니다**(PDF 원문이 출력에 남지 않게).
- 실행 위치는 어디여도 됩니다. 첫 셀이 저장소 루트를 찾아 경로를 잡습니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
for extra in (ROOT / "src", ROOT / "src" / "card_benefit"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import pymupdf

from payment_card.file_pipeline.doc_index.batch import select_documents

INDEX = ROOT / "src" / "card_benefit" / "data" / "index"
DATA = ROOT / "src" / "data"
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)
print("ROOT =", ROOT)

## 1. 인덱스 불러오기

In [ ]:
def load(issuer):
    docs = pd.read_csv(INDEX / issuer / "documents.csv", encoding="utf-8-sig", dtype=str).fillna("")
    links = pd.read_csv(INDEX / issuer / "product_docs.csv", encoding="utf-8-sig", dtype=str).fillna("")
    return docs, links

hana_docs, hana_links = load("hana")
woori_docs, woori_links = load("woori")
for name, docs, links in (("hana", hana_docs, hana_links), ("woori", woori_docs, woori_links)):
    print(name, "documents", len(docs), "product_docs", len(links))

## 2. 분포 확인 (문서 역할 / 카드 구분 / 대표 여부)

In [ ]:
for name, docs in (("hana", hana_docs), ("woori", woori_docs)):
    print("=====", name)
    print(docs.groupby(["card_type", "doc_role", "canonical"]).size().rename("files").reset_index().to_string(index=False))
    print("ocr_needed=Y:", (docs.ocr_needed == "Y").sum(), "/ needs_review=Y:", (docs.needs_review == "Y").sum())

## 3. 검토 대상 (사유별 건수와 목록)

In [ ]:
review = pd.concat([hana_docs.assign(issuer="hana"), woori_docs.assign(issuer="woori")])
review = review[review.needs_review == "Y"]
reasons = review.review_reason.str.split("; ").explode()
print(reasons.value_counts().to_string())
review[["issuer", "card_type", "doc_role", "review_reason", "file_path"]].head(20)

## 4. 배치 변환 대상 / 건너뜀(OCR) 확인

In [ ]:
for name, docs in (("hana", hana_docs), ("woori", woori_docs)):
    targets, skipped = select_documents(docs.to_dict("records"))
    print(name, "변환 대상", len(targets), "/ OCR로 건너뜀", len(skipped))
pd.DataFrame(skipped).head(10)

## 5. 샘플 PDF: 평문 vs 표 추출 비교

평문은 `traceability`가 저장하는 원문과 같은 방식입니다. 표 추출(`find_tables`)은 보조 정보 후보입니다. 아래 `SAMPLES`의 코드만 바꿔 다른 카드를 볼 수 있습니다.

In [ ]:
SAMPLES = [("hana", "15299"), ("hana", "11746"), ("woori", "102862"), ("woori", "500120"), ("woori", "500103")]

def find_pdf(issuer, code):
    hits = [p for p in (DATA / f"{issuer}_all" / "credit").glob(f"*{code}*") if "웹페이지" not in p.name]
    return hits[0]

rows = []
for issuer, code in SAMPLES:
    doc = pymupdf.open(find_pdf(issuer, code))
    for no, page in enumerate(doc, start=1):
        rows.append({"issuer": issuer, "code": code, "page": no,
                     "chars": len(page.get_text()), "tables": len(page.find_tables().tables)})
pd.DataFrame(rows)

In [ ]:
def show_page(issuer, code, page_no, max_chars=900):
    doc = pymupdf.open(find_pdf(issuer, code))
    page = doc[page_no - 1]
    print("--- 평문 (앞부분) ---")
    print(page.get_text()[:max_chars])
    for i, table in enumerate(page.find_tables().tables[:2], start=1):
        print(f"--- 표 {i}: {table.row_count}행 x {table.col_count}열 ---")
        for row in table.extract()[:6]:
            print([(cell or "").replace("\n", " ")[:16] for cell in row])

show_page("woori", "500103", 3)